In [11]:
import torch

print(torch.backends.mps.is_available())
print(torch.backends.mps.is_built())

True
True


In [12]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

embedder = SentenceTransformer(
    "intfloat/e5-large-v2",
    device=device
)

print("Using device:", device)

Using device: mps


In [14]:
import sys
sys.path.insert(0, '../..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import time
import json
import joblib
import mlflow
from pathlib import Path
from tqdm import tqdm

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    SparseVectorParams, SparseIndexParams,
    SparseVector, NamedVector, NamedSparseVector,
    SearchRequest, Filter, FieldCondition,
    MatchValue, OptimizersConfigDiff,
    HnswConfigDiff
)
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp

from src.utils.config import settings

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)


PROC = '../../data/processed/'
FEAT = '../../data/features/'

print(f"✅ Imports ready")
print(f"   Device : {device}")

✅ Imports ready
   Device : mps


In [15]:
client = QdrantClient(
    host = settings.QDRANT_HOST,
    port = settings.QDRANT_PORT,
)

# Test connection
collections = client.get_collections()
print(f"✅ Connected to Qdrant")
print(f"   Host        : {settings.QDRANT_HOST}:"
      f"{settings.QDRANT_PORT}")
print(f"   Collections : "
      f"{len(collections.collections)}")

# Qdrant version info
print(f"\n   Dashboard   : "
      f"http://localhost:6333/dashboard")

✅ Connected to Qdrant
   Host        : localhost:6333
   Collections : 0

   Dashboard   : http://localhost:6333/dashboard


In [16]:
# Load Data
movies  = pd.read_csv(PROC + 'movies_master.csv',
                      low_memory=False)
ratings = pd.read_csv(PROC + 'ratings_cleaned.csv')

# Parse list columns
import ast
def safe_parse(val):
    try:
        r = ast.literal_eval(str(val))
        return r if isinstance(r, list) else []
    except:
        return []

movies['genres_list']  = movies['genres_list']\
    .apply(safe_parse)
movies['cast_names']   = movies['cast_names']\
    .apply(safe_parse)
movies['keyword_list'] = movies['keyword_list']\
    .apply(safe_parse)

# Fill text fields
for col in ['title', 'overview', 'tagline',
            'director']:
    movies[col] = movies[col].fillna('')

# Only keep movies with movieId
movies = movies[movies['movieId'].notna()].copy()
movies['movieId'] = movies['movieId'].astype(int)

# Load movie interaction stats
mov_stats = pd.read_csv(
    FEAT + 'movie_interaction_features.csv')
movies = movies.merge(
    mov_stats[['movieId', 'rating_mean',
               'rating_count', 'popularity_tier']],
    on='movieId', how='left'
)

print(f"Movies to index : {len(movies):,}")
print(f"Columns         : {list(movies.columns)}")

Movies to index : 45,454
Columns         : ['id', 'title', 'original_title', 'overview', 'tagline', 'genres', 'release_date', 'year', 'original_language', 'budget', 'revenue', 'runtime', 'vote_average', 'vote_count', 'popularity', 'production_companies', 'poster_path', 'imdb_id', 'cast_names', 'director', 'keyword_list', 'movieId', 'tmdbId', 'genres_list', 'rating_mean', 'rating_count', 'popularity_tier']


In [17]:
# Build Text Soup For Embeddings

def build_text_soup(row) -> str:
    """
    Rich text representation per movie.
    Same as Day 4 but optimised for e5-large.
    e5-large works best with natural sentences
    not just keyword concatenation.
    """
    parts = []

    # Title — most important
    if row['title']:
        parts.append(f"Movie: {row['title']}.")

    # Overview — semantic content
    if row['overview']:
        parts.append(row['overview'])

    # Genres as sentence
    if row['genres_list']:
        genres = ', '.join(row['genres_list'])
        parts.append(f"Genre: {genres}.")

    # Director
    if row['director']:
        parts.append(
            f"Directed by {row['director']}.")

    # Cast
    if row['cast_names']:
        cast = ', '.join(row['cast_names'][:3])
        parts.append(f"Starring {cast}.")

    # Keywords
    if row['keyword_list']:
        kw = ', '.join(row['keyword_list'][:8])
        parts.append(f"Keywords: {kw}.")

    # Tagline
    if row['tagline']:
        parts.append(row['tagline'])

    return ' '.join(parts).strip()


movies['text_soup'] = movies.apply(
    build_text_soup, axis=1)

print(f"✅ Text soups built")
print(f"\nExample for '{movies['title'].iloc[0]}':")
print(movies['text_soup'].iloc[0][:400])
print(f"\nAvg length: "
      f"{movies['text_soup'].str.len().mean():.0f} "
      f"chars")

✅ Text soups built

Example for 'Toy Story':
Movie: Toy Story. Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences. Genre: Animation, Comedy, Family. Directed by John Lasseter. Starring Tom Hank

Avg length: 505 chars


In [18]:
# Load sentence-transformers e5-large
print("Loading sentence-transformers e5-large...")
print("First run downloads ~1.2GB model")
print("Subsequent runs load from cache\n")

start = time.time()

# e5-large: 2025-26 SOTA for text retrieval
# Better than all-MiniLM on MTEB benchmark
# Use 'query:' prefix for queries
# Use 'passage:' prefix for documents
model_name = 'intfloat/e5-large-v2'

embedder = SentenceTransformer(
    model_name, device=device)

elapsed = time.time() - start
print(f"✅ Model loaded in {elapsed:.1f}s")
print(f"   Model      : {model_name}")
print(f"   Embed dim  : "
      f"{embedder.get_sentence_embedding_dimension()}")
print(f"   Device     : {device}")

# Test embedding
test_emb = embedder.encode(
    ["passage: Test movie about space exploration"],
    normalize_embeddings=True
)
print(f"   Test embed : {test_emb.shape}")

Loading sentence-transformers e5-large...
First run downloads ~1.2GB model
Subsequent runs load from cache

✅ Model loaded in 4.4s
   Model      : intfloat/e5-large-v2
   Embed dim  : 1024
   Device     : mps
   Test embed : (1, 1024)


In [ ]:
# Generate Dense Embeddings
EMBED_DIM  = embedder\
    .get_sentence_embedding_dimension()
BATCH_SIZE = 64

print(f"Generating e5-large embeddings...")
print(f"  Movies     : {len(movies):,}")
print(f"  Embed dim  : {EMBED_DIM}")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  Device     : {device}\n")

# e5-large uses 'passage:' prefix for documents
texts = [
    f"passage: {soup}"
    for soup in movies['text_soup'].values
]

start      = time.time()
embeddings = embedder.encode(
    texts,
    batch_size        = BATCH_SIZE,
    show_progress_bar = True,
    normalize_embeddings = True,   # L2 normalise
    convert_to_numpy  = True,
)
elapsed = time.time() - start

print(f"\n✅ Embeddings generated")
print(f"   Shape   : {embeddings.shape}")
print(f"   Time    : {elapsed:.1f}s")
print(f"   Dtype   : {embeddings.dtype}")
print(f"   Norm    : "
      f"{np.linalg.norm(embeddings[0]):.4f} "
      f"(should be ~1.0)")

# Save embeddings
np.save(FEAT + 'e5_large_embeddings.npy',
        embeddings)
print(f"\n✅ Embeddings saved to "
      f"data/features/e5_large_embeddings.npy")

Generating e5-large embeddings...
  Movies     : 45,454
  Embed dim  : 1024
  Batch size : 64
  Device     : mps



Batches:   0%|          | 0/711 [00:00<?, ?it/s]

In [ ]:
# Build BM25 Sparse Vectors

